# A/B cierre few-shot — 2 turnos cerrados en el prefijo (v7 + prefijo)

- **Checkpoint**: el mismo merged del v7 (mismo base HF `BlinkDL/rwkv7-g1::rwkv7-g1j-2.9b-20260831-ctx16384.pth`, misma revisión `cd67fb95fa9e2ce8757f8d21d713e74a9c788118`, mismo adapter `out/kateto-final/adapter_model.safetensors`).
- **Parámetros**: `max_len=256`, T 0.80, top_p 0.70, seed 1337, `WKV=cuda`, mismos 60 items, mismo motor (`rwkv-peft nativo (.pth) / rwkvt RWKV7, WKV=cuda`).
- **Brazo A**: `render_chat([], p, "seco")` (idéntico al v7).
- **Brazo B**: `render_chat(HIST, p, "seco")` con `HIST` = los 2 turnos del `prefix.json`.
- **SHA-256 del prefijo**: `96c08793c56b5cbd6a70c9d7ac9eae55148d8c2e78d1076167cd1d4a26043b87`.
- **Nota**: El prefijo sale del set de ENTRENAMIENTO (`data/rwkv_kateto_seco.jsonl`, líneas 3668 y 4623) y no de ningún set de evaluación.


In [ ]:
import sys
import os
import json
import hashlib
import shutil
import importlib.util
import subprocess
from pathlib import Path

OUT = Path("/kaggle/working/out-eval29")
OUT.mkdir(parents=True, exist_ok=True)

STATE = {"env": {}, "stages": {}}

def log(*a):
    print("[eval29]", *a, flush=True)

def flush_state():
    (OUT / "manifest.json").write_text(json.dumps(STATE, ensure_ascii=False, indent=1), encoding="utf-8")

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while True:
            chunk = f.read(4 * 1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

# Deps defensivas
for mod_name, pip_name in [("einops", "einops"), ("yaml", "pyyaml"), ("triton", "triton"), ("huggingface_hub", "huggingface_hub")]:
    if importlib.util.find_spec(mod_name) is None:
        log(f"Instalando {pip_name}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)

import torch
import triton

n_gpu = torch.cuda.device_count()
gpu_names = [torch.cuda.get_device_name(i) for i in range(n_gpu)]
STATE["env"] = {
    "python": sys.version,
    "torch": torch.__version__,
    "triton": getattr(triton, "__version__", "unknown"),
    "n_gpu": n_gpu,
    "gpu": gpu_names,
}
log("ENV:", STATE["env"])
assert n_gpu >= 1, f"Se requiere al menos 1 GPU, encontrados: {n_gpu}"

WORK = None
for cand_str in ("/kaggle/temp/eval29", "/root/eval29"):
    cand = Path(cand_str)
    cand.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(cand).free >= 20 * (1024 ** 3):
        WORK = cand
        break

assert WORK is not None, "No se encontro workdir con >= 20 GiB libres en /kaggle/temp/eval29 o /root/eval29"
log(f"WORK dir: {WORK} (free: {shutil.disk_usage(WORK).free / (1024**3):.2f} GiB)")
STATE["work_dir"] = str(WORK)

flush_state()


In [ ]:
# CELL 2 — Stage del dataset desde /kaggle/input (layout agnostico: zip / tar / carpeta)
import glob, zipfile, tarfile

# 1. Localizar raiz del dataset de forma robusta
cands = glob.glob("/kaggle/input/**/merge_lora.py", recursive=True)
assert cands, "no encontre merge_lora.py en /kaggle/input"
SRC = Path(cands[0]).parent
log(f"SRC detectado: {SRC}")

# 2. Arbol de /kaggle/input (hasta 3 niveles, ~60 entradas)
print("arbol de /kaggle/input:")
vistos = 0
for _root, _dirs, _files in os.walk("/kaggle/input", followlinks=True):
    _prof = _root.replace("/kaggle/input", "").count("/")
    if _prof > 3:
        _dirs[:] = []
        continue
    for _n in sorted(_files)[:8]:
        print("   ", os.path.join(_root, _n))
        vistos += 1
    for _d in sorted(_dirs)[:8]:
        print("   ", os.path.join(_root, _d) + "/")
    if vistos > 60:
        break

# 3. PROJ
PROJ = WORK / "kateto"
PROJ.mkdir(parents=True, exist_ok=True)

# 4. Extraer *.zip recursivo
zips = sorted(glob.glob("/kaggle/input/**/*.zip", recursive=True))
for z in zips:
    base = os.path.basename(z)[:-4]
    with zipfile.ZipFile(z) as zf:
        names = zf.namelist()
    top = names[0].split("/")[0] if names else ""
    dest = str(PROJ) if top == base else os.path.join(str(PROJ), base)
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(dest)
    print(f"unzip {os.path.basename(z)} -> {dest} ({len(names)} entradas)")

# 5. Extraer *.tar recursivo
tars = sorted(glob.glob("/kaggle/input/**/*.tar", recursive=True))
for t in tars:
    with tarfile.open(t) as tf:
        names = tf.getnames()
    top = names[0].split("/")[0] if names else ""
    dest = str(PROJ) if top == os.path.basename(t)[:-4] else os.path.join(str(PROJ), top)
    os.makedirs(dest, exist_ok=True)
    with tarfile.open(t) as tf:
        tf.extractall(dest if top == os.path.basename(t)[:-4] else str(PROJ))
    print(f"untar {os.path.basename(t)} -> {dest} ({len(names)} entradas)")

# 6. Fallback: copiar archivos sueltos si faltan rutas requeridas o no habia archives
REQ_PATHS = [
    "scripts/eval_behavior.py",
    "scripts/eval_cbb_pmi.py",
    "scripts/eval_generation.py",
    "rwkv_pipeline/opd_rollout.py",
    "rwkv_pipeline/infer_kateto.py",
    "rwkv_pipeline/chat_template.py",
    "rwkv_pipeline/_stop_helpers.py",
    "RWKV-PEFT/rwkvt/rwkv7/model.py",
    "RWKV-PEFT/rwkv_vocab_v20230424.txt",
    "data/hf/che-boludo-benchmark/datasets/dataset_cbb_dpo.json",
    "out/eval/prompts_v1.json",
    "out/kateto-final/adapter_model.safetensors",
    "out/kateto-final/adapter_config.json",
    "config/behavior_patterns.yaml",
    "merge_lora.py",
    "data/eval_prompts.jsonl",
]
if not zips and not tars or any(not (PROJ / r).exists() for r in REQ_PATHS):
    log("Copiando archivos sueltos desde SRC...")
    for _p in glob.glob(str(SRC) + "/**/*", recursive=True):
        if os.path.isfile(_p):
            _rel = os.path.relpath(_p, str(SRC))
            _dst = PROJ / _rel
            _dst.parent.mkdir(parents=True, exist_ok=True)
            if not _dst.exists():
                shutil.copy2(_p, _dst)

# 7. Asserts
for r in REQ_PATHS:
    assert (PROJ / r).exists(), f"falta {r}"
log(f"Staging OK: {len(REQ_PATHS)} rutas verificadas en {PROJ}")

# 8. Estado
STATE["staging"] = {"src": str(SRC), "proj": str(PROJ), "n_checked": len(REQ_PATHS), "zips": len(zips), "tars": len(tars), "dataset_slug": "kateto-eval29"}
flush_state()


In [ ]:
from huggingface_hub import hf_hub_download

base_repo = "BlinkDL/rwkv7-g1"
base_filename = "rwkv7-g1j-2.9b-20260831-ctx16384.pth"
# g1j-2.9b fue borrado de main el 2026-10-08: se pinea la revision del baseline v5
BASE_REVISION = "cd67fb95fa9e2ce8757f8d21d713e74a9c788118"   # sha256 base 966f3420... = el del manifest v5
hf_dir = WORK / "hf"
hf_dir.mkdir(parents=True, exist_ok=True)
base_cand = hf_dir / base_filename

if not (base_cand.exists() and base_cand.stat().st_size > 5_500_000_000):
    log(f"Descargando {base_filename} desde HuggingFace ({base_repo})...")
    dl_path = hf_hub_download(
        repo_id=base_repo,
        filename=base_filename,
        local_dir=str(hf_dir),
        revision=BASE_REVISION,
    )
    BASE = Path(dl_path)
else:
    log(f"Base ya presente: {base_cand} ({base_cand.stat().st_size} bytes)")
    BASE = base_cand

base_bytes = BASE.stat().st_size
log(f"Calculando sha256 de base {BASE} ({base_bytes} bytes)...")
base_sha256 = sha256_file(BASE)
log(f"Base sha256: {base_sha256}")

STATE["base"] = {
    "path": str(BASE),
    "bytes": base_bytes,
    "sha256": base_sha256,
    "hf": f"{base_repo}::{base_filename}",
}
flush_state()


In [ ]:
MERGED = WORK / "kateto-rwkv29-merged.pth"
ADAPTER_DIR = PROJ / "out/kateto-final"
ADAPTER_WEIGHTS = ADAPTER_DIR / "adapter_model.safetensors"

if not (MERGED.exists() and MERGED.stat().st_size > 9_000_000_000):
    log("Iniciando merge_lora.py sobre base en CPU...")
    merge_cmd = [
        sys.executable,
        str(PROJ / "merge_lora.py"),
        str(BASE),
        str(ADAPTER_DIR),
        str(MERGED),
    ]
    res = subprocess.run(merge_cmd, cwd=str(PROJ))
    assert res.returncode == 0, f"merge_lora.py fallo con returncode={res.returncode}"
else:
    log(f"Merged ya existe con tamano suficiente: {MERGED} ({MERGED.stat().st_size} bytes)")

merged_bytes = MERGED.stat().st_size
log("Calculando sha256 de adapter...")
adapter_sha256 = sha256_file(ADAPTER_WEIGHTS)
log("Calculando sha256 de merged...")
merged_sha256 = sha256_file(MERGED)

STATE["adapter_sha256"] = adapter_sha256
STATE["merged"] = {
    "path": str(MERGED),
    "bytes": merged_bytes,
    "sha256": merged_sha256,
    "recipe": f"{BASE.name} + {ADAPTER_DIR.name} via merge_lora.py",
}

print(f"SHA256 base:    {STATE['base']['sha256']}")
print(f"SHA256 adapter: {STATE['adapter_sha256']}")
print(f"SHA256 merged:  {STATE['merged']['sha256']}")

flush_state()


In [ ]:
import time

def run_stage(name, argv, expect=()):
    log(f"STAGE {name} comenzando...")
    t0 = time.time()
    try:
        proc = subprocess.run(argv, cwd=str(PROJ))
        rc = proc.returncode
    except Exception as e:
        log(f"Excepcion corriendo stage {name}: {e}")
        rc = -1
    wall_s = round(time.time() - t0, 3)

    expect_paths = [Path(p) for p in expect]
    expect_ok = all(p.exists() for p in expect_paths)
    ok = (rc == 0) and expect_ok

    print(f"STAGE {name} rc={rc} wall={wall_s} ok={ok}", flush=True)
    if not expect_ok:
        missing = [str(p) for p in expect_paths if not p.exists()]
        log(f"Archivos esperados faltantes en {name}: {missing}")

    STATE["stages"][name] = {
        "argv": [str(x) for x in argv],
        "rc": rc,
        "wall_s": wall_s,
        "ok": ok,
    }
    flush_state()
    return ok

flush_state()


In [ ]:
import statistics
import os as _os
# Mismo motor que v5: rama CUDA de rwkvt/operator/rwkvop.py (no triton).
_os.environ["WKV"] = "cuda"  # antes de importar rwkv_pipeline (setdefault no pisa)
MOTOR = f"rwkv-peft nativo (.pth) / rwkvt RWKV7, WKV={_os.environ.get('WKV', '?')}"

pv1_path = PROJ / "out/eval/prompts_v1.json"
pv1_data = json.loads(pv1_path.read_text(encoding="utf-8"))
items = []
for it in pv1_data["generation"][:30]:
    items.append({
        "set": "prompts_v1_generation",
        "id": it["id"],
        "prompt": it["prompt"],
    })

ep_path = PROJ / "data/eval_prompts.jsonl"
ep_lines = [ln.strip() for ln in ep_path.read_text(encoding="utf-8").splitlines() if ln.strip()]
for i, ln in enumerate(ep_lines[:30]):
    row_data = json.loads(ln)
    items.append({
        "set": "eval_prompts_jsonl",
        "id": f"ep_{i:02d}",
        "prompt": row_data["prompt"],
    })

if str(PROJ) not in sys.path:
    sys.path.insert(0, str(PROJ))
import rwkv_pipeline.opd_rollout as orl
from scripts.eval_behavior import load_student, ensure_eval
from chat_template import render_chat, render_historial

model, tok, info = load_student(str(MERGED), device="cuda")
ensure_eval(model)
STATE["arch"] = info

PREFIX = json.loads(r"""{
  "schema": "eureka-fewshot-prefix/v1",
  "fuente": {
    "path": "data/rwkv_kateto_seco.jsonl",
    "sha256_archivo": "e424824679a807607fa2a4e1739e5b7e5bfa1641eaab43a740e8242fc4904416",
    "n_lineas_archivo": 4789,
    "lineas": [
      3668,
      4623
    ],
    "sha256_lineas": [
      "d20ae9319570e3e91a38d6c775e4c96fceed834c431fcdbe24fe03b980b61a61",
      "033ede1307c4045155effb53d062736ed676f6b77e0d66dc8a32e327697382ab"
    ]
  },
  "excluidos": {
    "path_eval_prompts": "data/eval_prompts.jsonl",
    "path_eval_v1": "out/eval/prompts_v1.json",
    "n_prompts": 60
  },
  "voice": "seco",
  "turnos": [
    [
      "dale, lo dejo correr",
      "boludo, ya lo estás tirando como si fuera un laburo sin fin."
    ],
    [
      "¿Qué es un servidor?",
      "Una máquina que se queda despierta esperando que alguien le pida algo."
    ]
  ],
  "prefijo_texto": "<|im_user|>dale, lo dejo correr<|im_end|>\n<|im_start|>seco\nboludo, ya lo estás tirando como si fuera un laburo sin fin.<|im_end|><|im_user|>¿Qué es un servidor?<|im_end|>\n<|im_start|>seco\nUna máquina que se queda despierta esperando que alguien le pida algo.<|im_end|>",
  "sha256_prefijo": "96c08793c56b5cbd6a70c9d7ac9eae55148d8c2e78d1076167cd1d4a26043b87",
  "n_chars_prefijo": 268,
  "marcadores": {
    "<|im_end|>": 4,
    "<|im_user|>": 2,
    "<|im_start|>": 2
  },
  "nota_anti_hack": "Los marcadores los emite unicamente el template (render_turn/render_historial). slot_guard verifica que ni el slot user ni el slot answer contienen marcadores, asi que el <|im_end|> del ejemplo vive SOLO en el turno del assistant: no hay marcador dentro del prompt del user."
}""")
HIST = [tuple(t) for t in PREFIX["turnos"]]
PREFIJO_TXT = PREFIX["prefijo_texto"]
assert render_historial(HIST, "seco") == PREFIJO_TXT, "El historial no coincide con prefijo_texto"
assert PREFIX["sha256_prefijo"] == hashlib.sha256(PREFIJO_TXT.encode("utf-8")).hexdigest(), "SHA256 del prefijo no coincide"

n_prefijo_tokens = len(tok.encode(PREFIJO_TXT))
log(f"Prefijo validado: {len(HIST)} turnos, {len(PREFIJO_TXT)} chars, {n_prefijo_tokens} tokens")

def fmt(p, arm):
    if arm == "A":
        return render_chat([], p, "seco")
    elif arm == "B":
        return render_chat(HIST, p, "seco")
    raise ValueError(f"Brazo desconocido: {arm}")

_ORIG = orl._next_logits

def make_timer():
    box = {"n": 0, "t0": None, "ttfab": None}
    def wrapped(*args, **kwargs):
        if box["n"] == 0:
            box["t0"] = time.time()
        res = _ORIG(*args, **kwargs)
        if box["n"] == 0:
            box["ttfab"] = round(time.time() - box["t0"], 4)
        box["n"] += 1
        return res
    return box, wrapped

def check_degenerate(text):
    t = text.strip()
    if len(t) < 8:
        return False, 0
    reps = 0
    for n in (8, 12, 16, 24):
        for i in range(0, max(0, len(t) - 2 * n)):
            sub = t[i:i + n]
            c = t.count(sub)
            if c > reps:
                reps = c
    return (reps >= 3), reps

dev_str = "cuda0:" + torch.cuda.get_device_name(0)

def gen_row(item, turno, arm, rows_file):
    box, wrapped = make_timer()
    orl._next_logits = wrapped
    try:
        torch.manual_seed(1337)
        prompt_txt = item["prompt"]
        prompt_formatted = fmt(prompt_txt, arm)
        r = orl.rollout_from_prompt(
            model, tok, prompt_formatted,
            max_len=256, temperature=0.80, top_p=0.70
        )
        text = orl._decode_str(tok, list(r.response_ids))
        is_deg, max_rep = check_degenerate(text)
        row = {
            "set": item["set"],
            "id": item["id"],
            "prompt": prompt_txt,
            "prompt_sha256": hashlib.sha256(prompt_txt.encode("utf-8")).hexdigest(),
            "model": "kateto-rwkv29-merged.pth",
            "motor": MOTOR,
            "engine": "rwkvt.rwkv7.model.RWKV7",
            "device": dev_str,
            "turno": turno,
            "chat_format": "chatml",
            "arm": arm,
            "prefix_sha256": PREFIX["sha256_prefijo"] if arm == "B" else None,
            "n_prefijo_tokens": n_prefijo_tokens if arm == "B" else 0,
            "ttfab_s": box["ttfab"],
            "wall_s": r.wall_s,
            "n_tokens": len(r.response_ids),
            "finished": bool(r.finished),
            "stop_reason": getattr(r, "stop_reason", None),
            "degenerate": is_deg,
            "max_repeat": max_rep,
            "text": text[:4000],
            "error": None,
        }
    except Exception as e:
        prompt_txt = item["prompt"]
        row = {
            "set": item["set"],
            "id": item["id"],
            "prompt": prompt_txt,
            "prompt_sha256": hashlib.sha256(prompt_txt.encode("utf-8")).hexdigest(),
            "model": "kateto-rwkv29-merged.pth",
            "motor": MOTOR,
            "engine": "rwkvt.rwkv7.model.RWKV7",
            "device": dev_str,
            "turno": turno,
            "chat_format": "chatml",
            "arm": arm,
            "prefix_sha256": PREFIX["sha256_prefijo"] if arm == "B" else None,
            "n_prefijo_tokens": n_prefijo_tokens if arm == "B" else 0,
            "ttfab_s": box["ttfab"],
            "wall_s": None,
            "n_tokens": 0,
            "finished": False,
            "stop_reason": None,
            "degenerate": False,
            "max_repeat": 0,
            "text": "",
            "error": str(e),
        }
    finally:
        orl._next_logits = _ORIG

    with open(rows_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()
    return row

for arm in ("A", "B"):
    arm_dir = OUT / f"arm-{arm}"
    arm_dir.mkdir(parents=True, exist_ok=True)
    rows_file = arm_dir / "gen29_rows.jsonl"
    if rows_file.exists():
        rows_file.unlink()

    log(f"=== INICIANDO BRAZO {arm} ===")
    arm_rows = []
    dummy_item = {"set": "dummy", "id": "dummy_00", "prompt": "hola"}
    r_dummy = gen_row(dummy_item, "frio_proceso", arm, rows_file)
    arm_rows.append(r_dummy)

    for idx, it in enumerate(items):
        rf = gen_row(it, "frio", arm, rows_file)
        rc = gen_row(it, "caliente", arm, rows_file)
        arm_rows.extend([rf, rc])
        if (idx + 1) % 10 == 0:
            log(f"[Brazo {arm}] Generados {idx + 1}/{len(items)} items...")

    assert len(arm_rows) == 121, f"Brazo {arm}: esperado 121 filas, obtenido {len(arm_rows)}"
    if arm == "B":
        tokens_set = {r["n_prefijo_tokens"] for r in arm_rows}
        assert len(tokens_set) == 1 and list(tokens_set)[0] > 0, f"Brazo B: tokens_set {tokens_set} invalido"
    else:
        assert all(r["n_prefijo_tokens"] == 0 for r in arm_rows), "Brazo A: tokens no son todos 0"

    val_frio = [r["ttfab_s"] for r in arm_rows if r["turno"] == "frio" and r["ttfab_s"] is not None and not r["error"]]
    val_caliente = [r["ttfab_s"] for r in arm_rows if r["turno"] == "caliente" and r["ttfab_s"] is not None and not r["error"]]
    med_frio = round(statistics.median(val_frio), 4) if val_frio else None
    med_caliente = round(statistics.median(val_caliente), 4) if val_caliente else None

    prefix_block = {k: v for k, v in PREFIX.items() if k != "prefijo_texto"} if arm == "B" else None
    arm_manifest = {
        "arm": arm,
        "prefix": prefix_block,
        "merged": STATE.get("merged"),
        "base": STATE.get("base"),
        "adapter_sha256": STATE.get("adapter_sha256"),
        "arch": STATE.get("arch"),
        "sampling": {
            "max_len": 256,
            "temperature": 0.80,
            "top_p": 0.70,
            "seed": 1337,
            "ngram_block": None,
        },
        "gen_rows": {
            "n": len(arm_rows),
            "median_ttfab_frio": med_frio,
            "median_ttfab_caliente": med_caliente,
        },
        "n_prefijo_tokens": n_prefijo_tokens if arm == "B" else 0,
    }
    (arm_dir / "manifest.json").write_text(json.dumps(arm_manifest, ensure_ascii=False, indent=1), encoding="utf-8")

STATE["arms"] = {
    "arm-A": {"path": str(OUT / "arm-A" / "gen29_rows.jsonl"), "n_filas": 121},
    "arm-B": {"path": str(OUT / "arm-B" / "gen29_rows.jsonl"), "n_filas": 121, "n_prefijo_tokens": n_prefijo_tokens, "sha256_prefijo": PREFIX["sha256_prefijo"]},
}

from scripts.eval_behavior import free_gpu
free_gpu(model)

flush_state()


In [ ]:
REFERENCE_V7_SHA256 = "e4f36a26fbbb5f6c45acb0c1c11944dc540bed0f279a756bfc9d43ea254eb065"

sha_base = STATE.get("base", {}).get("sha256", "unknown")
sha_adapter = STATE.get("adapter_sha256", "unknown")
sha_merged = STATE.get("merged", {}).get("sha256", "unknown")

summary_ab = f"""# Resumen A/B Cierre Few-Shot

## Artefactos
- **Base SHA-256**: `{sha_base}`
- **Adapter SHA-256**: `{sha_adapter}`
- **Merged SHA-256**: `{sha_merged}`
- **Reference v7 SHA-256**: `{REFERENCE_V7_SHA256}`

## Prefijo Few-Shot
- **Prefijo SHA-256**: `{PREFIX['sha256_prefijo']}`
- **Prefijo tokens**: `{n_prefijo_tokens}`

## Filas Generadas
- **Brazo A**: `{OUT / "arm-A" / "gen29_rows.jsonl"}` (121 filas)
- **Brazo B**: `{OUT / "arm-B" / "gen29_rows.jsonl"}` (121 filas)
"""

(OUT / "SUMMARY-AB.md").write_text(summary_ab, encoding="utf-8")
log(f"SUMMARY-AB.md escrito en {OUT / 'SUMMARY-AB.md'}")
print("A/B DONE", flush=True)
flush_state()
